In [1]:
import asyncio
from agent_framework import WorkflowBuilder, WorkflowContext, executor, AgentResponseUpdate
from agent_framework.observability import configure_otel_providers
from typing_extensions import Never
from utils import new_base_openai_agent, new_base_gemini_agent
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import time
import json
from telemetry import configure_ipc_telemetry

# Note: Workflow events are a live application stream; 
# exported traces describe operations that have finished and are usually sent in batches.

def show_telemetry(b: bytes) -> None:
    payload = json.loads(b)
    signal = payload["signal"]
    records = payload["records"]
    for record in records:
        print(f"{time.time()}: {signal} - {record}")

configure_ipc_telemetry(show_telemetry)

In [3]:
@executor(id="upper_case")
async def upper_case(text: str, ctx: WorkflowContext[str, str]):
    await ctx.send_message(text.upper())
    await ctx.yield_output('Upper Case is done')

@executor(id="reverse_text")
async def reverse_text(text: str, ctx: WorkflowContext[Never, str]) -> None:
    await asyncio.sleep(1)
    await ctx.yield_output(text[::-1])

workflow = (
    WorkflowBuilder(start_executor=upper_case)
    .add_edge(upper_case, reverse_text)
    .build()
)

In [5]:
async for event in workflow.run("hello world", stream=True):
    print(event)

WorkflowEvent(type='started')
WorkflowEvent(type='status', state=IN_PROGRESS)
WorkflowEvent(type='superstep_started', iteration=1)
WorkflowEvent(type='executor_invoked', executor_id='upper_case', data='hello world')
WorkflowEvent(type='output', executor_id='upper_case', data='Upper Case is done')
WorkflowEvent(type='executor_completed', executor_id='upper_case', data=['HELLO WORLD', 'Upper Case is done'])
WorkflowEvent(type='superstep_completed', iteration=1)
WorkflowEvent(type='superstep_started', iteration=2)
WorkflowEvent(type='executor_invoked', executor_id='reverse_text', data='HELLO WORLD')
WorkflowEvent(type='output', executor_id='reverse_text', data='DLROW OLLEH')
WorkflowEvent(type='executor_completed', executor_id='reverse_text', data=['DLROW OLLEH'])
WorkflowEvent(type='superstep_completed', iteration=2)
WorkflowEvent(type='status', state=IDLE)


1793796105.7728884: spans - {'name': 'edge_group.process InternalEdgeGroup', 'context': {'trace_id': '0x35a1ba7cc86e34018c909279bae840bb', 'span_id': '0xb8f50e3f7e74e0f7', 'trace_state': '[]'}, 'kind': 'SpanKind.INTERNAL', 'parent_id': '0x709bdc925b25dd93', 'start_time': '2026-11-04T12:41:44.608009Z', 'end_time': '2026-11-04T12:41:44.608028Z', 'status': {'status_code': 'UNSET'}, 'attributes': {'edge_group.type': 'InternalEdgeGroup', 'edge_group.id': 'InternalEdgeGroup/9c417924-4c09-4fa6-a104-630df61a000d', 'message.source_id': 'internal:upper_case', 'message.target_id': 'upper_case', 'edge_group.delivered': True, 'edge_group.delivery_status': 'delivered'}, 'events': [], 'links': [], 'resource': {'attributes': {'telemetry.sdk.language': 'python', 'telemetry.sdk.name': 'opentelemetry', 'telemetry.sdk.version': '1.45.2', 'service.instance.id': '5eba7501-d4f8-4faf-a263-191b0bfb52f6', 'microsoft.agent-framework': 'True', 'service.name': 'agent_framework', 'service.version': '1.24.0'}, 'sche

## Telemetry 101

Telemetry generally has three signals:

| Signal | Purpose | Example |
|---|---|---|
| **Traces** | Follow an operation across components | Workflow → executor → model request |
| **Logs** | Record individual messages or occurrences | “Request failed” |
| **Metrics** | Aggregate numerical measurements | Request count, token usage, duration histogram |

A **trace** consists of related **spans**. Each span represents an operation with a start time, end time, attributes, and possibly errors or timestamped events.

Workflow events arrive in realtime, while spans are recorded and only send when they have completed. Child spans can export before the parent finishes. OpenTelemetry does not need to wait for the entire trace—only for each individual span to end.

**A span represents one operation. A trace is the collection of related spans describing a larger operation end to end.**

### Example

Running a workflow produces several spans:

```text
Trace: one workflow execution
│
└── workflow.run                    ← root span
    ├── executor.process upper_case ← child span
    │   └── message.send            ← child of the executor span
    └── executor.process reverse_text
```

Every line naming an operation is a **span**. The entire collection is the **trace**.

The root span represents the overall workflow operation; **it is not itself the whole trace**.

### What a span contains

A span records information about its particular operation:

- **Name:** what operation ran.
- **Start and end timestamps:** when it ran and how long it took.
- **Attributes:** details such as executor ID or model name.
- **Status:** for example, whether an error occurred.
- **Events:** timestamped annotations during the operation.
- **IDs:** which trace it belongs to and which span is its parent.

### How they connect

Consider simplified IDs:

| Span | `trace_id` | `span_id` | `parent_span_id` |
|---|---|---|---|
| Workflow | `T1` | `A` | None |
| Upper-case executor | `T1` | `B` | `A` |
| Message send | `T1` | `C` | `B` |
| Reverse-text executor | `T1` | `D` | `A` |

- The shared **`trace_id`** groups them into one trace.
- Each **`span_id`** identifies an individual operation.
- The **parent span ID** establishes the hierarchy.

Another independent workflow execution would normally have a different trace ID.

### What your IPC exporter receives

Your span exporter receives **individual completed spans, grouped into export batches**—not necessarily a complete trace.

```text
First export:   spans B and C
Later export:   spans D and A
```

The Go process can group them by `trace_id` and reconstruct their relationships using the parent IDs. A batch can also contain spans from multiple traces.

This explains the terminology in your payload:

```json
{
  "signal": "traces",
  "records": ["individual span records go here"]
}
```

“Traces” names the telemetry category; **spans are the records being transported**.


### What starts a new trace? 

The framework decides, but typically a new span without an active parent span will create a new trace.

Start a span
│
├── Valid parent context exists
│   → inherit its trace ID
│   → create a new span ID
│   → become a child of that parent
│
└── No valid parent context exists
    → generate a new trace ID
    → create a new span ID
    → become the root span

We can create our own traces with the following:

```python
from opentelemetry import trace

tracer = trace.get_tracer("my-application")

with tracer.start_as_current_span("handle_user_request"):
    async for event in workflow.run("hello world", stream=True):
        print(event)
```


In [6]:
# LEFT-OFF: reviewing options to reduce latency of telemetry data export (see chat window)
# LEFT-OFF: understanding how SimpleSpanProcessor works and differs from my current setup